# Wake — 尾场: TESLA 模块尾场表

本 notebook 是官方算例系列示例之一 (全部位于 examples/ 目录): 只跑这一个官方算例, 逐步解读输入卡、展示本算例最有代表性的图。

In [ ]:
%run ../notebooks/_bootstrap.py

In [ ]:
# ===== 共享规格 (单一数据源, 与 06 汇总一致) =====
from examples._examples_spec import (
    EXAMPLES, run_example, phase_files, compare_xemit)
NAME = "Wake"
STEM = "Wake"
print("算例:", NAME, "| 物理:", EXAMPLES[NAME]["title"])


In [ ]:
# ===== 运行本算例 (generator/astra 按需) =====
work = run_example(NAME)
print("输出文件:")
for f in sorted(work.glob(STEM + ".*")):
    print("  ", f.name)

In [ ]:
# ===== 束团统计 (最后一个 z 位置) =====
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html
ph = phase_files(work, STEM)
dist = read_distribution(ph[-1])
print("相空间文件:", ph[-1].name)
print_statistics(compute_statistics(dist),
                 title="%s @ %s" % (NAME, ph[-1].name))
stats_table_html(compute_statistics(dist))

from astra_tools.io.field_map import read_wake_potential
import matplotlib.pyplot as plt
w = read_wake_potential(work / "TESLA_MODULE_WAKE_TAYLOR.dat")
print("尾场块数:", len(w.blocks))
for bi, blk in enumerate(w.blocks):
    tag = "单极 (monopole)" if bi == 0 else "双极 (dipole %d)" % bi
    print("  block %d: %d 点 (%s)" % (bi + 1, len(blk.s), tag))
    plt.plot(blk.s * 1e3, blk.w, lw=0.8, label="block %d %s" % (bi + 1, tag))
plt.xlabel("s [mm]")
plt.ylabel("W [V/C]")
plt.title("TESLA 模块尾场表")
plt.legend(fontsize=8)
plt.tight_layout()


In [ ]:
from astra_tools.analysis.bff import compute_bff
from astra_tools.plot.bff_plots import plot_bff_with_amplitude
bff = compute_bff(dist.filter_active().z, dist.filter_active().charge,
                  kmin=10, kmax=1e5, nk=150, detect_features=True)
plot_bff_with_amplitude(bff)

In [ ]:
# ===== 黄金比对 (rel < 0.5% 判 OK) =====
compare_xemit(NAME, work)
print("其他官方算例的示例 notebook 同样位于 examples/ 目录")